# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AzlanFaisalRaj/flyrank-internship-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

I rank the honest-split model's test rows (`work/notebooks/w06_validation_audit.ipynb`, client-grouped logistic regression) by `priority_score = model_probability x impressions_90d` — a page only earns a high rank if the model flags real decline risk **and** enough people actually see it. A 0.9-probability page with 5 impressions ranks below a 0.6-probability page with 300,000 impressions, on purpose: this is a review queue for a human with limited time, not a leaderboard of raw model confidence.

**Reason codes** (any row can carry several):
- `model_decline_risk` — model probability >= 0.6
- `visible_traffic` — impressions_90d >= 300 (someone is actually seeing this page)
- `stale_content` — freshness_tier is 91-180 days (the window Week-4's signal check found elevated, page-6 of the paper's own decay-curve finding backs this direction too)
- `ctr_below_expected` — CTR under the position-tier's own median (same `expected_ctr_by_tier` definition as the Week-4 baseline, so this playbook and that baseline agree on what "underperforming" means)
- `low_engagement` — engagement_rate under its position-tier median
- `no_flag` — none of the above; nothing here to review

**Archetype -> action mapping:**

| Reason codes present | Action |
|---|---|
| model risk < 0.5, or not visible | `monitor` — not enough evidence or not enough traffic to justify effort |
| `ctr_below_expected` + `low_engagement` | `refresh_and_review_ctr_and_engagement` — two independent gaps, worth a fuller look |
| `ctr_below_expected` only | `refresh_and_review_ctr` — likely a snippet/title/intent-match problem |
| `low_engagement` only | `refresh_and_review_engagement` — page structure/readability problem, not a click problem |
| `stale_content` only (no CTR/engagement gap) | `refresh` — a routine freshness pass, no specific symptom flagged |
| none of the above | `monitor` |

**The decay/refresh insight this leans on:** the FlyRank paper's Finding #4 measured a 3.2x health-score boost and 57x impression boost when 365+ day content was refreshed within 30 days, and Finding #2 measured the decay cliff starting around 271-365 days. My own Week-4 signal check found the `91-180` freshness window specifically elevated (61.1% decline rate vs a 54.2% base rate) in this starter dataset. Both point the same direction — staleness matters, but the specific window matters more than "old = bad" — so `stale_content` alone is a weaker signal than `stale_content` combined with a measured CTR or engagement gap, and the action mapping above reflects that: staleness by itself gets the lightest action (`refresh`), while staleness plus a measured gap earns the fuller `refresh_and_review_*` treatment.


In [1]:
import pandas as pd, numpy as np, os
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

RANDOM_SEED = 42
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

window_cols = [c for c in df.columns if c.endswith("_last_30d") or c.endswith("_prev_30d")]
drop_cols = {"content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label"} | set(window_cols)
feature_cols = [c for c in df.columns if c not in drop_cols]
num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]
cat_cols = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(df[c])]

# same client-grouped, honest split as w06
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
train, test = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

cat_pipe = Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))])
pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_cols),
    ("cat", cat_pipe, cat_cols),
])
model = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_SEED))])
model.fit(train[feature_cols], train["is_declining_label"])
test["model_proba"] = model.predict_proba(test[feature_cols])[:, 1]

# business signals — same definitions as the Week-4 baseline rule, so the two agree
floor = train[train["impressions_90d"] >= 300]
expected_ctr_by_tier = floor.groupby("position_tier")["ctr"].median()
test["expected_ctr"] = test["position_tier"].map(expected_ctr_by_tier)
test["stale"] = test["freshness_tier"] == "91-180"
test["visible"] = test["impressions_90d"] >= 300
test["ctr_gap"] = test["ctr"] < test["expected_ctr"]
test["low_engagement"] = test["engagement_rate"] < test.groupby("position_tier")["engagement_rate"].transform("median")

def reason_codes(row):
    codes = []
    if row["model_proba"] >= 0.6: codes.append("model_decline_risk")
    if row["visible"]: codes.append("visible_traffic")
    if row["stale"]: codes.append("stale_content")
    if row["ctr_gap"] and row["visible"]: codes.append("ctr_below_expected")
    if row["low_engagement"] and row["visible"]: codes.append("low_engagement")
    return ",".join(codes) if codes else "no_flag"

def action_label(row):
    if row["model_proba"] < 0.5 or not row["visible"]:
        return "monitor"
    if row["ctr_gap"] and row["low_engagement"]:
        return "refresh_and_review_ctr_and_engagement"
    if row["ctr_gap"]:
        return "refresh_and_review_ctr"
    if row["low_engagement"]:
        return "refresh_and_review_engagement"
    if row["stale"]:
        return "refresh"
    return "monitor"

test["reason_code"] = test.apply(reason_codes, axis=1)
test["action_label"] = test.apply(action_label, axis=1)
test["priority_score"] = (test["model_proba"] * test["impressions_90d"]).round(1)

ranked = test.sort_values("priority_score", ascending=False).reset_index(drop=True)
ranked["rank"] = ranked.index + 1

print("Action mix:")
print(ranked["action_label"].value_counts())
print()
print("Top 5 reason-code combinations:")
print(ranked["reason_code"].value_counts().head(5))
print()
out_cols = ["rank","content_id","client_id","priority_score","model_proba","action_label",
            "reason_code","impressions_90d","ctr","expected_ctr","position_tier","freshness_tier","engagement_rate"]
print("Top 10 of the ranked queue:")
ranked[out_cols].head(10)


Action mix:
action_label
monitor                   5522
refresh_and_review_ctr    1369
refresh                    224
Name: count, dtype: int64

Top 5 reason-code combinations:
reason_code
no_flag                                                  1786
model_decline_risk                                       1255
model_decline_risk,visible_traffic,ctr_below_expected     810
visible_traffic                                           774
model_decline_risk,visible_traffic                        702
Name: count, dtype: int64

Top 10 of the ranked queue:


,rank,content_id,client_id,priority_score,model_proba,action_label,reason_code,impressions_90d,ctr,expected_ctr,position_tier,freshness_tier,engagement_rate
0,1,content_5fe46e04994d,client_4e07408562,343853.0,0.664174,refresh_and_review_ctr,"model_decline_risk,visible_traffic,stale_conte...",517715,0.14,0.24,page_1,91-180,4.23
1,2,content_c84a0ab98e90,client_f369cb89fc,193936.7,0.868616,refresh_and_review_ctr,"model_decline_risk,visible_traffic,ctr_below_e...",223271,0.03,0.24,page_1,0-30,3.45
2,3,content_73c54f78c06a,client_f369cb89fc,164727.6,0.769888,refresh_and_review_ctr,"model_decline_risk,visible_traffic,ctr_below_e...",213963,0.10,0.24,page_1,0-30,0.86
3,4,content_8c19996aa890,client_4e07408562,158621.4,0.311479,monitor,"visible_traffic,ctr_below_expected",509252,0.15,0.23,top_3,0-30,11.73
4,5,content_db5989a78dd3,client_4e07408562,150752.0,0.436822,monitor,"visible_traffic,ctr_below_expected",345111,0.21,0.24,page_1,0-30,2.32
5,6,content_2db251d1a841,client_f369cb89fc,141505.7,0.712261,refresh_and_review_ctr,"model_decline_risk,visible_traffic,ctr_below_e...",198671,0.18,0.24,page_1,0-30,4.98
6,7,content_cea79ef51519,client_f369cb89fc,130724.7,0.626082,refresh_and_review_ctr,"model_decline_risk,visible_traffic,ctr_below_e...",208798,0.23,0.24,page_1,0-30,2.87
7,8,content_453722754fea,client_f369cb89fc,126953.1,0.906296,refresh_and_review_ctr,"model_decline_risk,visible_traffic,ctr_below_e...",140079,0.01,0.24,page_1,0-30,0.00
8,9,content_8451fc6f034d,client_d029fa3a95,118489.2,0.435392,monitor,"visible_traffic,ctr_below_expected",272144,0.03,0.23,top_3,0-30,2.02
9,10,content_39881853ef0c,client_f369cb89fc,97667.0,0.868661,refresh_and_review_ctr,"model_decline_risk,visible_traffic,ctr_below_e...",112434,0.01,0.24,page_1,0-30,3.45


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use.** A content strategist or SEO reviewer running a weekly or biweekly refresh triage session, using this ranked queue as a **starting shortlist** to look at first, out of a much larger content library they don't have time to review page by page. The output is a prioritized reading list with a stated reason for each entry, not a final decision.

**Limits, stated plainly:**

- **One static CSV snapshot.** All numbers come from a single 90-day-window export (`data/raw/content_refresh_anonymized.csv`, 30,000 rows, 32 clients). There is no time-series validation here — the honest split is client-grouped, not time-aware, because this starter file has no real timestamp column to split on (see `w06_validation_audit.ipynb`, Section 2).
- **Observed patterns, not causal claims.** Per `writing-honest-claims/SKILL.md`: this is cross-sectional data, so nothing here says refreshing a page *will cause* it to recover. The honest form is decision-support — "this page looks worth reviewing first, because X."
- **Model performance is modest.** precision@10 = 0.80 and ROC AUC = 0.575 on the honest client-grouped split (from `w06_validation_audit.ipynb`) — a real lift over the 0.517 base rate and the Week-4 rule baseline, but far from a reliable oracle. 2 in 10 of the top-10 picks are expected to be wrong even under the model's own best-case numbers.
- **32 clients only, uneven traffic mix.** The Week-4 baseline notebook found 6 of its top 10 rule-based picks concentrated in just 2 clients, because `impressions_90d` drives the score. The same risk applies here: this queue can structurally favor high-traffic clients over clients with a smaller but more urgent problem relative to their own size.
- **No query-level or SERP-feature context.** The dataset doesn't say *why* a CTR is low — a title/meta problem and a SERP feature stealing the click above the result look identical in these columns.


In [2]:
print("Honest-split model metrics this playbook is built on (from w06_validation_audit.ipynb):")
print(f"  precision@10 = 0.800   (base rate = {df['is_declining_label'].mean():.3f})")
print(f"  ROC AUC      = 0.575")
print(f"  client overlap between train/test = 0  (client-grouped split)")
print()
print("Clients represented in this queue's top 20:")
print(ranked.head(20)["client_id"].value_counts())


Honest-split model metrics this playbook is built on (from w06_validation_audit.ipynb):
  precision@10 = 0.800   (base rate = 0.542)
  ROC AUC      = 0.575
  client overlap between train/test = 0  (client-grouped split)

Clients represented in this queue's top 20:
client_id
client_4e07408562    11
client_f369cb89fc     8
client_d029fa3a95     1
Name: count, dtype: int64


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any row, a human must check:**
1. Open the actual page. Confirm the content genuinely reads as dated, thin, or off-intent — don't trust the reason code alone.
2. For `ctr_below_expected` rows: rule out a branded/navigational query (users don't click through by nature there), a mid-run title/meta experiment, and a SERP feature (snippet, People Also Ask) sitting above the result.
3. For any row with `ctr == 0.00%` at real volume: treat as a possible tracking/pipeline gap first, a content problem second — a genuine zero at high impressions is unusual enough to warrant a manual check (same flag the Week-4 baseline notebook raised).
4. Check for near-duplicate content within the same client — two rows can describe the same underlying page cluster, which would double-count one real action as two.

**What should NOT be automated — full no-go list:**
- **No auto-publishing.** Nothing in this queue should trigger an automatic content edit, republish, or metadata change. Every action needs a human editor.
- **No automatic deprioritization / removal.** A `monitor` label means "not enough evidence right now," never "delete" or "deprioritize forever."
- **No cross-client comparison as a performance judgment.** Because the queue is traffic-weighted, a client appearing rarely in the top ranks is not evidence that client's content is fine — see the concentration limit above.
- **No use as a client-facing report without review.** Reason codes and probabilities are internal triage language (`writing-honest-claims/SKILL.md`'s decision-support standard), not client-ready claims about why a page underperforms.
- **No retraining or threshold changes without a human sign-off.** Section 4 below defines *when* to look again — not permission to change the model or thresholds automatically.


In [3]:
zero_ctr_flag = ranked[(ranked["ctr"] == 0) & (ranked["impressions_90d"] >= 300)]
print(f"Rows flagged for the 'possible tracking gap' check (CTR=0.00% at impressions_90d>=300): {len(zero_ctr_flag)}")
print(zero_ctr_flag[["content_id","client_id","impressions_90d","ctr","position_tier"]].head(5).to_string(index=False))


Rows flagged for the 'possible tracking gap' check (CTR=0.00% at impressions_90d>=300): 734
          content_id         client_id  impressions_90d  ctr position_tier
content_df71843dcd17 client_8527a891e2            27334  0.0          deep
content_8ba781dafa55 client_8527a891e2            16156  0.0        page_1
content_825a9788af8d client_4e07408562            16786  0.0        page_1
content_c82bc0c24241 client_f369cb89fc            13676  0.0        page_1
content_5d5653c4eb4f client_4e07408562            15101  0.0        page_1


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

- **Base-rate drift.** If the portfolio's actual decline rate moves meaningfully away from this snapshot's 54.2% (e.g. a new content push, a algorithm update), the model's calibration is stale — recheck before trusting probabilities.
- **Precision@10 drop on a fresh audit sample.** Periodically pull a new labeled sample and recompute precision@10 against it. A drop toward the 0.517 base rate means the model has stopped adding value over guessing.
- **Client-mix shift.** If the client roster changes (new clients added, one client's traffic share grows sharply), the client-grouped validation no longer represents the current mix — retrain and re-validate on the new group composition.
- **Feature distribution shift.** Watch `word_count`, `content_age_days`, and `freshness_tier` distributions month over month; a sharp shift (e.g. a bulk content migration, like the same-day staleness cluster the Week-4 notebook flagged) means the model is scoring content unlike what it was trained on.
- **Reason-code mix collapse.** If `no_flag` starts covering an unusually large or small share of rows versus this run's baseline, the underlying signals (CTR-vs-tier medians, engagement medians) likely need recomputing on fresher data.
- **Retrain cadence (starting point, not a rule):** quarterly, or immediately if any trigger above fires — whichever comes first.


In [4]:
print("Baseline figures for future drift checks (this run):")
print(f"  base rate (decline):        {df['is_declining_label'].mean():.3f}")
print(f"  reason_code mix (this run):")
print(ranked["reason_code"].value_counts(normalize=True).round(3).head(6))
print(f"  clients represented:        {df['client_id'].nunique()}")


Baseline figures for future drift checks (this run):
  base rate (decline):        0.542
  reason_code mix (this run):
reason_code
no_flag                                                  0.251
model_decline_risk                                       0.176
model_decline_risk,visible_traffic,ctr_below_expected    0.114
visible_traffic                                          0.109
model_decline_risk,visible_traffic                       0.099
visible_traffic,ctr_below_expected                       0.080
Name: proportion, dtype: float64
  clients represented:        32


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Per the assignment note, the queue CSV is regenerated by this notebook and stays out of git (the CI leak-guard blocks data files in `work/outputs/`). The metrics JSON and the two figures below are committed — they're the receipts this playbook's numbers, and next week's paper, trace back to.


In [5]:
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# 1. Ranked queue CSV -> work/outputs/ (gitignored by design, regenerated each run)
ranked[out_cols].to_csv("work/outputs/w07_action_playbook_queue.csv", index=False)
print("Wrote", len(ranked), "rows to work/outputs/w07_action_playbook_queue.csv")

# 2. Metrics JSON -> work/outputs/ (COMMITTED — the receipts)
metrics = {
    "model": "logistic_regression_grouped_split",
    "split": "client_grouped (GroupShuffleSplit, test_size=0.25, seed=42)",
    "base_rate": round(float(df["is_declining_label"].mean()), 3),
    "precision_at_10": 0.800,
    "precision_at_20": 0.700,
    "roc_auc": 0.575,
    "n_test_rows": int(len(test)),
    "n_clients_total": int(df["client_id"].nunique()),
    "action_mix": ranked["action_label"].value_counts().to_dict(),
    "top_reason_codes": ranked["reason_code"].value_counts().head(6).to_dict(),
}
with open("work/outputs/w07_playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Wrote work/outputs/w07_playbook_metrics.json")
print(json.dumps(metrics, indent=2))

# 3. Figures -> work/figures/ (COMMITTED)
fig1, ax1 = plt.subplots(figsize=(6,4))
ranked["action_label"].value_counts().plot(kind="barh", ax=ax1, color="#2b6cb0")
ax1.set_title("Action mix — Week-7 playbook queue")
ax1.set_xlabel("count")
fig1.tight_layout()
fig1.savefig("work/figures/w07_action_mix.png", dpi=120)
plt.close(fig1)

fig2, ax2 = plt.subplots(figsize=(6,4))
ranked["reason_code"].value_counts().head(8).plot(kind="barh", ax=ax2, color="#2f855a")
ax2.set_title("Top reason-code combinations")
ax2.set_xlabel("count")
fig2.tight_layout()
fig2.savefig("work/figures/w07_reason_codes.png", dpi=120)
plt.close(fig2)

print("Wrote work/figures/w07_action_mix.png and work/figures/w07_reason_codes.png")


Wrote 7115 rows to work/outputs/w07_action_playbook_queue.csv
Wrote work/outputs/w07_playbook_metrics.json
{
  "model": "logistic_regression_grouped_split",
  "split": "client_grouped (GroupShuffleSplit, test_size=0.25, seed=42)",
  "base_rate": 0.542,
  "precision_at_10": 0.8,
  "precision_at_20": 0.7,
  "roc_auc": 0.575,
  "n_test_rows": 7115,
  "n_clients_total": 32,
  "action_mix": {
    "monitor": 5522,
    "refresh_and_review_ctr": 1369,
    "refresh": 224
  },
  "top_reason_codes": {
    "no_flag": 1786,
    "model_decline_risk": 1255,
    "model_decline_risk,visible_traffic,ctr_below_expected": 810,
    "visible_traffic": 774,
    "model_decline_risk,visible_traffic": 702,
    "visible_traffic,ctr_below_expected": 570
  }
}


Wrote work/figures/w07_action_mix.png and work/figures/w07_reason_codes.png


## 6. Five minute demo outline (optional, Week 8 showcase)

If presenting, here is a tight five minute walkthrough of this capstone.

**Question (30 seconds).** Out of thousands of content pages, which ones should a content strategist review first for a refresh, given limited time each week?

**Method (1 minute).** Compared a transparent staleness and CTR gap rule against a client grouped logistic regression, on 30,000 real FlyRank content items across 32 clients. Split by client, not by row, so the model is never tested on a client it already learned from. Ran a deliberate leakage test by adding the excluded window columns back in, confirming the model's advantage is real and not a leak.

**One chart (1.5 minutes).** Show the precision at 10 and precision at 20 bar chart (`figures/capstone_model_vs_baseline.png`): the rule sits at 0.30, the base rate at 0.52, and the logistic regression model reaches 0.80.

**One honest result (1 minute).** The model clears both the rule and the base rate at the top of the queue, but ROC AUC only reaches about 0.575 to 0.60. That means the model is a genuine improvement for prioritizing a short review list, not a confident, near certain predictor of decline.

**One recommendation (1 minute).** Use the ranked queue as a starting shortlist, always open the actual page before acting on a reason code, and never automate publishing, editing, or deprioritization from this output alone.


## 7. Two shareable cuts

**Social post (methodology focused).**

I built a model to answer a question every content team faces: which pages should get reviewed first? Using 30,000 real search performance records across 32 clients, I compared a simple, explainable rule against a client grouped logistic regression, tested on clients the model never saw during training. The rule barely beat guessing at precision@10 (0.30). The model reached 0.80, and a leakage test confirmed the gap was real, not a shortcut. The output is a ranked, reason coded review queue, built for a human to check, not to act on blindly.

**Employer facing summary (3 sentences).**

I built a client grouped logistic regression that ranks content pages by refresh priority, trained and evaluated on 30,000 real search performance records across 32 clients from a production content warehouse. Tested against a transparent rule baseline on an identical, client grouped holdout split, the model lifted precision at the top of the queue from 0.30 to 0.80, and I confirmed the gap with a dedicated leakage check rather than taking it at face value. The result is a decision support tool: a ranked, reason coded shortlist for a content strategist, with explicit limitations and a clear list of actions that should never be automated from it.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.